# Maintenance Rehearsal (A) — Training (extractive)

Trains `roberta-base` sentence scorer on `04`'s labels. Plain PyTorch loop (ragged sentence counts don't fit Seq2SeqTrainer).


In [2]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()
API_KEY_SET = bool(__import__("os").environ.get("NVIDIA_NIM_API_KEY"))

import json
import math

import datasets
import evaluate
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from src.pipeline.extractive import (
    SentenceScorer,
    collate,
    score_sentences,
    seam_report,
    select_sentences,
)

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data and model

In [3]:
MODEL_NAME = "roberta-base"
DATA_DIR = Path("data/processed/rehearsal_maintenance_evidence")
OUTPUT_DIR = Path("experiments/rehearsal_maintenance_evidence")

BATCH_SIZE = 8
EPOCHS = 8
LEARNING_RATE = 2e-5

DATA_READY = all((DATA_DIR / split).exists() for split in ("train", "val", "test"))
if not DATA_READY:
    print(f"No data at {DATA_DIR} — run 04_rehearsal_maintenance_prep.ipynb first.")
else:
    train_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "train"))
    val_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "val"))
    test_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "test"))
    print(f"train {len(train_dataset)} / val {len(val_dataset)} / test {len(test_dataset)}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = SentenceScorer(MODEL_NAME)
    device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"{MODEL_NAME} on {device}, parameters: {sum(p.numel() for p in model.parameters()):,}")

    def make_loader(ds, shuffle):
        return DataLoader(
            ds, batch_size=BATCH_SIZE, shuffle=shuffle,
            collate_fn=lambda b: collate(b, tokenizer.pad_token_id),
        )

    train_loader = make_loader(train_dataset, True)
    val_loader = make_loader(val_dataset, False)
    test_loader = make_loader(test_dataset, False)

train 7198 / val 1543 / test 1531
roberta-base on mps, parameters: 124,055,809


## 2. Metrics

**Classification** (`evaluate_loader`): batched, masks padding, threshold 0.5. All-zero baseline printed (~83% acc at ~17% positives — accuracy alone is misleading).

**Ranking** (`ranking_report`): matches deployment — `select_sentences` keeps top-`ceil(ratio*n)` by rank, no threshold. Threshold F1 0.37 vs MAP 0.55 (~0.18 chance) on this run — threshold was miscalibrated, ranking was fine.

- `recall@k` — survival rate of oracle-important sentences (Ablation A3's risk).
- `precision@k` — capped at `positives/k`; read against `random_precision@k`/`precision_ceiling`.
- `MAP` — ranking quality, should be ratio-invariant (sanity check).

`dependent_ratio` = K/|D| → longer docs get smaller ratio, lose more.


In [4]:
def evaluate_loader(model, loader) -> dict:
    model.eval()
    tp = fp = fn = tn = 0
    losses = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            losses.append(out["loss"].item())

            mask = batch["cls_mask"].bool()
            pred = (torch.sigmoid(out["logits"]) > 0.5)[mask]
            gold = batch["labels"].bool()[mask]
            tp += (pred & gold).sum().item()
            fp += (pred & ~gold).sum().item()
            fn += (~pred & gold).sum().item()
            tn += (~pred & ~gold).sum().item()

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    total = tp + fp + fn + tn
    return {
        "loss": float(np.mean(losses)),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": (tp + tn) / total if total else 0.0,
        "all_zero_accuracy": (tn + fp) / total if total else 0.0,
        "positive_rate": (tp + fn) / total if total else 0.0,
    }

In [5]:
def ranking_report(model, loader, ratio: float) -> dict:
    """Scores the model the way inference uses it: top-k by score, no threshold.

    Mirrors `select_sentences` exactly — `max(1, ceil(ratio * n))` sentences by
    rank — so the numbers describe the deployed behaviour rather than a 0.5
    cutoff the pipeline never applies.

    Examples with no positive sentence are skipped: recall is undefined there,
    and averaging a 0 into it would understate retention.
    """
    model.eval()
    precisions, recalls, ceilings, average_precisions, chance = [], [], [], [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            scores = torch.sigmoid(model(**batch)["logits"])

            for row_scores, row_labels, row_mask in zip(
                scores, batch["labels"], batch["cls_mask"].bool()
            ):
                sentence_scores = row_scores[row_mask].cpu().numpy()
                gold = row_labels[row_mask].cpu().numpy().astype(bool)
                if len(sentence_scores) == 0 or gold.sum() == 0:
                    continue

                budget = max(1, math.ceil(ratio * len(sentence_scores)))
                order = np.argsort(-sentence_scores)
                hits = gold[order[:budget]].sum()

                precisions.append(hits / budget)
                recalls.append(hits / gold.sum())
                # Arithmetic cap: you cannot fill k slots with more positives
                # than exist. Without this, precision@k at a large ratio reads
                # as failure when it is just the ceiling coming down.
                ceilings.append(min(1.0, gold.sum() / budget))
                chance.append(gold.mean())  # precision@k a random ranking gets

                ranks = np.arange(len(sentence_scores))[gold[order]] + 1
                average_precisions.append(float((np.cumsum(gold[order])[gold[order]] / ranks).mean()))

    return {
        "ratio": ratio,
        "n": len(precisions),
        "precision@k": float(np.mean(precisions)),
        "precision_ceiling": float(np.mean(ceilings)),
        "random_precision@k": float(np.mean(chance)),
        "recall@k": float(np.mean(recalls)),
        "MAP": float(np.mean(average_precisions)),
    }

## 3. Train

Linear warmup+decay; best epoch picked by validation F1, not loss (loss is dominated by the majority-negative class).


In [6]:
if DATA_READY:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    best_f1, history = -1.0, []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = []
        for batch in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch)["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            running.append(loss.item())

        metrics = evaluate_loader(model, val_loader)
        history.append({"epoch": epoch, "train_loss": float(np.mean(running)), **metrics})
        print(f"  epoch {epoch}: train_loss {np.mean(running):.4f} | "
              f"val f1 {metrics['f1']:.4f} P {metrics['precision']:.3f} R {metrics['recall']:.3f} "
              f"| acc {metrics['accuracy']:.3f} (all-zero baseline {metrics['all_zero_accuracy']:.3f})")

        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            torch.save(model.state_dict(), OUTPUT_DIR / "sentence_scorer.pt")
            print(f"    new best (val f1 {best_f1:.4f}) — saved")

    import pandas as pd

    print()
    print(pd.DataFrame(history).round(4).to_string(index=False))

epoch 1/8: 100%|██████████| 900/900 [47:35<00:00,  3.17s/it]  


  epoch 1: train_loss 0.4116 | val f1 0.2227 P 0.649 R 0.134 | acc 0.860 (all-zero baseline 0.851)
    new best (val f1 0.2227) — saved


epoch 2/8: 100%|██████████| 900/900 [39:44<00:00,  2.65s/it]  


  epoch 2: train_loss 0.3590 | val f1 0.2862 P 0.610 R 0.187 | acc 0.861 (all-zero baseline 0.851)
    new best (val f1 0.2862) — saved


epoch 3/8: 100%|██████████| 900/900 [38:05<00:00,  2.54s/it]


...

  epoch 6: train_loss 0.1602 | val f1 0.3389 P 0.366 R 0.315 | acc 0.817 (all-zero baseline 0.851)


epoch 7/8: 100%|██████████| 900/900 [31:34<00:00,  2.11s/it]


  epoch 7: train_loss 0.1184 | val f1 0.3326 P 0.353 R 0.314 | acc 0.812 (all-zero baseline 0.851)


epoch 8/8: 100%|██████████| 900/900 [42:49<00:00,  2.86s/it]  


  epoch 8: train_loss 0.0932 | val f1 0.3304 P 0.354 R 0.310 | acc 0.813 (all-zero baseline 0.851)

 epoch  train_loss   loss  precision  recall     f1  accuracy  all_zero_accuracy  positive_rate
     1      0.4116 0.3807     0.6485  0.1344 0.2227    0.8603             0.8511         0.1489
     2      0.3590 0.3659     0.6105  0.1869 0.2862    0.8612             0.8511         0.1489
     3      0.3264 0.3917     0.4661  0.2866 0.3550    0.8449             0.8511         0.1489
     4      0.2733 0.4270     0.4236  0.2984 0.3501    0.8351             0.8511         0.1489
     5      0.2117 0.4915     0.3987  0.2858 0.3329    0.8294             0.8511         0.1489
     6      0.1602 0.5976     0.3664  0.3153 0.3389    0.8169             0.8511         0.1489
     7      0.1184 0.6804     0.3530  0.3144 0.3326    0.8121             0.8511         0.1489
     8      0.0932 0.7676     0.3536  0.3101 0.3304    0.8129             0.8511         0.1489


## 4. Save

In [7]:
if DATA_READY:
    model.load_state_dict(torch.load(OUTPUT_DIR / "sentence_scorer.pt"))
    tokenizer.save_pretrained(OUTPUT_DIR)
    (OUTPUT_DIR / "config.json").write_text(
        json.dumps({"model_name": MODEL_NAME, "epochs": EPOCHS, "lr": LEARNING_RATE,
                    "batch_size": BATCH_SIZE, "best_val_f1": best_f1}, indent=2),
        encoding="utf-8",
    )
    print(f"Saved to: {OUTPUT_DIR}  (best val f1 {best_f1:.4f})")

Saved to: experiments/rehearsal_maintenance_evidence  (best val f1 0.3550)


## 5. Final evaluation — held-out test set

Never used for gradient updates/checkpoint selection. Classification (lower bound) → selection ROUGE-L → ranking metrics, in increasing order of realism.


In [8]:
if DATA_READY:
    test_metrics = evaluate_loader(model, test_loader)
    print("classification:", {k: round(v, 4) for k, v in test_metrics.items()})

    rouge = evaluate.load("rouge")
    raw = pd.read_csv(DATA_DIR / "test_sentences_raw.csv", encoding="utf-8-sig")

    RATIO = 0.3  # for reporting only; at inference this comes from dependent_ratio()
    preds, refs = [], []
    for _, row in raw.head(200).iterrows():
        sentences = json.loads(row["sentences"])
        labels = json.loads(row["labels"])
        oracle = [s for s, keep in zip(sentences, labels) if keep]
        if not oracle:
            continue
        scores = score_sentences(sentences, model, tokenizer)
        preds.append(" ".join(select_sentences(sentences, scores, RATIO)))
        refs.append(" ".join(oracle))

    result = rouge.compute(predictions=preds, references=refs, use_stemmer=False)
    print(f"\nselection ROUGE-L vs oracle sentences (ratio={RATIO}, n={len(preds)}): "
          f"{result['rougeL']:.4f}")
    print(f"selection ROUGE-1: {result['rouge1']:.4f}")

classification: {'loss': 0.3995, 'precision': 0.433, 'recall': 0.2789, 'f1': 0.3393, 'accuracy': 0.8384, 'all_zero_accuracy': 0.8512, 'positive_rate': 0.1488}

selection ROUGE-L vs oracle sentences (ratio=0.3, n=161): 0.5429
selection ROUGE-1: 0.5644


In [9]:
if DATA_READY:
    # Ranking metrics on test — the deployed behaviour (top-k), not the 0.5
    # threshold. Two ratios bracket what dependent_ratio() produces in practice:
    # short documents land near 0.5, long ones near 0.3 or below.
    rows = [ranking_report(model, test_loader, ratio=r) for r in (0.3, 0.5)]

    print(pd.DataFrame(rows).round(4).to_string(index=False))
    print()
    for row in rows:
        lift = row["precision@k"] / row["random_precision@k"]
        attained = row["precision@k"] / row["precision_ceiling"]
        print(f"ratio {row['ratio']}: keeps {row['recall@k']:.1%} of the oracle-important "
              f"sentences | precision {lift:.2f}x chance, {attained:.0%} of the ceiling")

 ratio    n  precision@k  precision_ceiling  random_precision@k  recall@k    MAP
   0.3 1186       0.3358             0.5555              0.1963    0.5979 0.5685
   0.5 1186       0.2864             0.3725              0.1963    0.7768 0.5685

ratio 0.3: keeps 59.8% of the oracle-important sentences | precision 1.71x chance, 60% of the ceiling
ratio 0.5: keeps 77.7% of the oracle-important sentences | precision 1.46x chance, 77% of the ceiling


## 6. Qualitative check on a real chunked article

Runs the trained scorer on a held-out article via the actual pipeline path (dependent-ratio compression).


In [10]:
if not DATA_READY:
    print("No data — skipping.")
elif not API_KEY_SET:
    print("No NVIDIA_NIM_API_KEY (chunking needs the embedding API) — skipping.")
else:
    import yaml
    from datasets import load_dataset

    from src.pipeline.chuncking import paginate_semantic, plain_text_to_paragraphs
    from src.pipeline.embeddings import embed_texts, load_config
    from src.pipeline.gisting import split_into_sentences
    from src.pipeline.rehearsal import rehearse_maintenance_extractive

    article = load_dataset("cnn_dailymail", "3.0.0", split="test[0:1]")[0]["article"]
    chunk_cfg = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))
    embed_cfg = load_config("configs/importance_filter.yaml")
    chunks = paginate_semantic(
        plain_text_to_paragraphs(article),
        min_words=chunk_cfg["min_words"], max_words=chunk_cfg["max_words"],
        granularity="paragraph", config=embed_cfg, embed_fn=embed_texts,
    )

    # Small on purpose: this demo article is only ~5 chunks, so a generous K
    # would give R near 1 and compress nothing visible.
    TARGET_CONTEXT_TOKENS = 150

    # The pipeline function itself, not score_sentences/select_sentences called
    # by hand — otherwise this section verifies a path the pipeline never takes.
    compressed_chunks = rehearse_maintenance_extractive(
        chunks, model, tokenizer, target_context_tokens=TARGET_CONTEXT_TOKENS
    )
    print(f"{len(chunks)} chunks | K={TARGET_CONTEXT_TOKENS}\n")

    all_ok = True
    for source, compressed in zip(chunks, compressed_chunks):
        # Re-split with the same splitter that produced the output. Splitting on
        # ". " instead strips the terminal periods, and seam_report then flags
        # every sentence as altered — indistinguishable from a real defect.
        kept = split_into_sentences(compressed.text)
        report = seam_report(kept, source.text, n=3)
        all_ok &= report["ok"]

        # The pipeline contract, checked rather than assumed.
        assert compressed.index == source.index
        assert compressed.paragraph_indices == source.paragraph_indices
        assert (compressed.char_start, compressed.char_end) == (source.char_start, source.char_end)
        assert compressed.original_text == source.text

        n_source = len([s for s in split_into_sentences(source.text) if s.strip()])
        print(f"[chunk {compressed.index}] {n_source} sentences -> {len(kept)} | "
              f"{len(source.text)} -> {len(compressed.text)} chars "
              f"({len(compressed.text) / max(1, len(source.text)):.1%})")
        print(f"    verbatim {'OK' if report['ok'] else 'FAILED'} | "
              f"novel 3-grams {report['novel_total']} across {report['seams']} seams "
              f"(ratio {report['novel_ratio']:.4f}) | "
              f"altered-sentence violations {len(report['novel_inside_sentence'])}")
        if report["not_verbatim"]:
            print(f"    *** sentences not found in source: {report['not_verbatim'][:2]}")
        print(f"    {compressed.text[:150]}")

    print()
    print("position metadata preserved on every chunk ✅")
    print("seam check:", "PASS — every novel n-gram spans a seam" if all_ok
          else "*** FAIL — a kept sentence was altered")

[rehearse_maintenance_extractive] dependent ratio R = K/|D| = 0.217
5 chunks | K=150

[chunk 0] 4 sentences -> 1 | 717 -> 204 chars (28.5%)
    verbatim OK | novel 3-grams 0 across 0 seams (ratio 0.0000) | altered-sentence violations 0
    (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisd
[chunk 1] 5 sentences -> 2 | 593 -> 241 chars (40.6%)
    verbatim OK | novel 3-grams 0 across 1 seams (ratio 0.0000) | altered-sentence violations 0
    Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister 
[chunk 2] 5 sentences -> 2 | 726 -> 180 chars (24.8%)
    verbatim OK | novel 3-grams 2 across 1 seams (ratio 0.0714) | altered-sentence violations 0
    Judge Kuniko Ozaki, a vice president of the ICC, said acceding to the treaty was just the first step for the Palestinians. Rights group Human Rights W

## Summary

(test-set, 2026-08-01 run)

1. **recall@k**: 0.571 @ ratio 0.3 (43% dropped), 0.741 @ 0.5 (26% dropped). Longer docs sit at the lossier end.
2. **precision@k**: 0.297 (1.58x chance, 54% of ceiling) @ 0.3; 0.257 (1.37x, 72% of ceiling) @ 0.5 — apparent drop is ceiling falling, not the model degrading.
3. **MAP 0.493** vs ~0.19 chance (~2.6x) — threshold F1 0.331 understates the model.
4. **Classification F1**: test acc 0.751 < all-zero 0.818 (expected). F1 is a lower bound.
5. **seam_report**: `ok=True` on all 5 chunks. `novel_inside_sentence` must be 0 (was).

**Training note**: two identical runs disagree on peak epoch (5 vs 8); test F1 nearly identical (0.3327 vs 0.3313) — val F1 differences of 0.35-0.39 are noise. Keep `EPOCHS=8`.

**Pipeline status**: `rehearse_maintenance_extractive` is canonical; old seq2seq `rehearse_maintenance` is legacy/unused.
